# TTM vs Traditional Forecasters — Synthetic DGP Benchmark

Tests TTM against traditional forecasters on synthetic time series where the data-generating process is known.  
**Goal:** find settings where TTM is competitive or wins — and document where it doesn't.

| DGP | Classical advantage? | Why TTM might win |
|-----|---------------------|-------------------|
| ARIMA(3,3) | Yes (misspecified AR order helps classical) | — |
| GARCH-like volatility clustering | Partial | TTM sees raw residuals, may capture clustering |
| Regime-switching (low/high mean) | No | TTM sees long context, adapts faster |
| Nonlinear AR (logistic) | No | Classical AR is misspecified |


In [ ]:
import sys, os, logging

GRANITE_DIR = os.path.dirname(os.path.abspath('__file__'))
TIMESFM_DIR = os.path.abspath(os.path.join(GRANITE_DIR, '..', 'timesfm_google'))

for p in [GRANITE_DIR, TIMESFM_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

class _FreqTokenFilter(logging.Filter):
    def filter(self, record):
        return 'Frequency token' not in record.getMessage()
logging.getLogger().addFilter(_FreqTokenFilter())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm
from statsmodels.tsa.arima_process import ArmaProcess

from tsfm_public import (
    TimeSeriesForecastingPipeline,
    TimeSeriesPreprocessor,
    TinyTimeMixerForPrediction,
)
from benchmark import (
    BenchmarkResults,
    NaiveBenchmarkForecaster,
    ARIMAForecaster,
    AutoARIMAForecaster,
    BayesianARForecaster,
    MLBayesARForecaster,
    SSAForecaster,
)

device = (
    'mps'  if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else
    'cuda' if torch.cuda.is_available() else 'cpu'
)
print(f'device = {device}')

## Config

In [ ]:
N_OBS          = 2000   # total synthetic series length
K_FIRST        = 360    # initial training window
HORIZON        = 1      # steps ahead
TTM_CONTEXT    = 512    # max context for TTM
TRAD_MAX_HIST  = 512    # cap for traditional forecasters (fair comparison)
ROLLING_WINDOW = 36     # window for rolling MSE chart
SEED           = 42

DATE_COL   = 'date'
TARGET_COL = 'y'

## DGP definitions

Pick one DGP to run, or loop over all of them.

In [ ]:
rng = np.random.default_rng(SEED)

def make_arima33(n, rng):
    """ARIMA(3,3) — classical models have structural advantage here."""
    ar = np.array([1, -0.6, 0.2, -0.1])   # AR coefficients (lag-polynomial form)
    ma = np.array([1,  0.4, 0.3,  0.2])   # MA coefficients
    proc = ArmaProcess(ar, ma)
    return proc.generate_sample(nsample=n, scale=1.0, distrvs=rng.standard_normal)

def make_regime_switch(n, rng):
    """Two-regime AR(1) with random switches — classical models are misspecified."""
    regimes  = [{'mu': 0.0, 'phi': 0.9, 'sigma': 0.5},
                {'mu': 5.0, 'phi': 0.5, 'sigma': 1.5}]
    p_switch = 0.02   # prob of switching regime each step
    regime   = 0
    y = np.zeros(n)
    y[0] = regimes[regime]['mu']
    for t in range(1, n):
        if rng.random() < p_switch:
            regime = 1 - regime
        r  = regimes[regime]
        y[t] = r['mu'] + r['phi'] * (y[t-1] - r['mu']) + r['sigma'] * rng.standard_normal()
    return y

def make_nonlinear_ar(n, rng):
    """Logistic-map AR — highly nonlinear, classical linear models fail."""
    y = np.zeros(n)
    y[0] = rng.uniform(0.1, 0.9)
    r = 3.7   # chaos parameter
    for t in range(1, n):
        y[t] = r * y[t-1] * (1 - y[t-1]) + 0.05 * rng.standard_normal()
    return y

def make_garch(n, rng):
    """GARCH(1,1) — volatility clustering. Classical AR is badly misspecified."""
    omega, alpha, beta = 0.1, 0.15, 0.80
    y  = np.zeros(n)
    h  = np.ones(n)
    for t in range(1, n):
        h[t] = omega + alpha * y[t-1]**2 + beta * h[t-1]
        y[t] = np.sqrt(h[t]) * rng.standard_normal()
    return y

DGPS = {
    'ARIMA(3,3)':       lambda: make_arima33(N_OBS, rng),
    'Regime-switch AR': lambda: make_regime_switch(N_OBS, rng),
    'Nonlinear AR':     lambda: make_nonlinear_ar(N_OBS, rng),
    'GARCH(1,1)':       lambda: make_garch(N_OBS, rng),
}

# ── Pick which DGP to run ─────────────────────────────────────────────────
ACTIVE_DGP = 'ARIMA(3,3)'   # change to any key above

values = DGPS[ACTIVE_DGP]()
dates  = pd.date_range('2000-01-01', periods=N_OBS, freq='D')
df     = pd.DataFrame({DATE_COL: dates, TARGET_COL: values})

print(f'DGP: {ACTIVE_DGP}  |  n = {N_OBS}')
df.plot(x=DATE_COL, y=TARGET_COL, figsize=(14, 3), title=ACTIVE_DGP)
plt.tight_layout(); plt.show()

## Initialise models

In [ ]:
# ── TTM ──────────────────────────────────────────────────────────────────
_ctx         = min(TTM_CONTEXT, K_FIRST)
_init_ctx_df = df.iloc[K_FIRST - _ctx : K_FIRST].copy()

tsp = TimeSeriesPreprocessor(
    timestamp_column=DATE_COL,
    target_columns=[TARGET_COL],
    context_length=_ctx,
    prediction_length=max(HORIZON, 1),
    scaling=True,
    encode_categorical=False,
    scaler_type='standard',
)
tsp = tsp.train(_init_ctx_df)

ttm_model = TinyTimeMixerForPrediction.from_pretrained(
    'ibm-granite/granite-timeseries-ttm-r2',
    revision='512-96-ft-r2.1',
    num_input_channels=tsp.num_input_channels,
)
print('TTM ready')

# ── Traditional forecasters ───────────────────────────────────────────────
traditional = {
    'Naive':     NaiveBenchmarkForecaster(),
    'AR(1)':     ARIMAForecaster(order=(1, 0, 0)),
    'AutoARIMA': AutoARIMAForecaster(p_max=4, q_max=4, d_max=1),
    'BayesAR':   BayesianARForecaster(p=3, prior_precision=1.0, prior_mode='ridge'),
    'MLBayesAR': MLBayesARForecaster(p=24, prior_mode='minnesota'),
    'SSA':       SSAForecaster(),
}
print(f'Traditional forecasters: {list(traditional.keys())}')

## Rolling-window OOS loop

In [ ]:
origins      = list(range(K_FIRST, N_OBS - HORIZON + 1))
actuals      = []
origin_dates = []
ttm_preds    = []
trad_preds   = {name: [] for name in traditional}

for k in tqdm(origins, desc=f'Rolling OOS [{ACTIVE_DGP}]'):
    actuals.append(values[k + HORIZON - 1])
    origin_dates.append(dates[k])

    # ── TTM ──────────────────────────────────────────────────────────────
    ctx_len = min(TTM_CONTEXT, k)
    ctx_df  = df.iloc[k - ctx_len : k].copy().reset_index(drop=True)

    tsp_k = TimeSeriesPreprocessor(
        timestamp_column=DATE_COL,
        target_columns=[TARGET_COL],
        context_length=ctx_len,
        prediction_length=max(HORIZON, 1),
        scaling=True,
        encode_categorical=False,
        scaler_type='standard',
    )
    tsp_k = tsp_k.train(ctx_df)

    pipeline_k = TimeSeriesForecastingPipeline(
        ttm_model,
        device=device,
        feature_extractor=tsp_k,
        batch_size=512,
        add_known_ground_truth=False,
    )
    try:
        fcast    = pipeline_k(ctx_df)
        pred_col = f'{TARGET_COL}_prediction' if f'{TARGET_COL}_prediction' in fcast.columns else TARGET_COL
        pred_arr = np.asarray(fcast.iloc[-1][pred_col], dtype=float)
        ttm_preds.append(pred_arr[HORIZON - 1])
    except Exception:
        ttm_preds.append(np.nan)

    # ── Traditional — capped at TRAD_MAX_HIST ─────────────────────────────
    history = values[max(0, k - TRAD_MAX_HIST) : k]
    for name, fc in traditional.items():
        try:
            pred_arr = fc.fit_predict(history, HORIZON)
            trad_preds[name].append(float(pred_arr[HORIZON - 1]))
        except Exception:
            trad_preds[name].append(np.nan)

actuals      = np.array(actuals, dtype=float)
ttm_preds    = np.array(ttm_preds, dtype=float)
origin_dates = pd.to_datetime(origin_dates)
trad_preds   = {k: np.array(v, dtype=float) for k, v in trad_preds.items()}

print(f'Done. Origins: {len(origins)}')

## Wrap into BenchmarkResults & diagnostics

In [ ]:
all_preds = {'TTM (granite)': ttm_preds, **trad_preds}
results = BenchmarkResults(
    series_name=ACTIVE_DGP,
    horizon=HORIZON,
    k_first=K_FIRST,
    forecaster_names=list(all_preds.keys()),
    predictions={k: v.reshape(-1, HORIZON) for k, v in all_preds.items()},
    actuals=actuals.reshape(-1, HORIZON),
)
print(f'BenchmarkResults ready — {results.n_origins} origins, {len(results.forecaster_names)} forecasters')

In [ ]:
_sc = results.summary().copy()

_naive_row = _sc[_sc['Forecaster'] == 'Naive']
if not _naive_row.empty:
    _naive_rmse = float(_naive_row['RMSE'].iloc[0])
    _sc['RMSE/Naive'] = _sc['RMSE'] / _naive_rmse if _naive_rmse > 0 else float('nan')
else:
    _sc['RMSE/Naive'] = float('nan')

_num = [c for c in _sc.columns if c != 'Forecaster']
try:
    display(
        _sc.style
        .format({c: '{:.5f}' for c in _num}, na_rep='—')
        .set_properties(subset=_num, **{'text-align': 'right'})
        .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}], overwrite=False)
        .background_gradient(subset=['RMSE/Naive'], cmap='RdYlGn_r')
    )
except Exception:
    display(_sc.round(5))

print('\n── Diebold-Mariano (SE loss) ──')
display(results.diebold_mariano().round(4))

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')

COLORS = {
    'TTM (granite)': 'steelblue',
    'Naive':         'gray',
    'AR(1)':         'green',
    'AutoARIMA':     'darkorange',
    'BayesAR':       'purple',
    'MLBayesAR':     'crimson',
    'SSA':           'saddlebrown',
}

forecaster_names = results.forecaster_names
_x      = np.arange(len(forecaster_names))
_width  = 0.35
_maes   = [float(results.metrics('mae').set_index('Forecaster').loc[n, 'MAE']) for n in forecaster_names]
_mses   = [float(results.metrics('mse').set_index('Forecaster').loc[n, 'MSE']) for n in forecaster_names]
_ratios = _sc.set_index('Forecaster').loc[forecaster_names, 'RMSE/Naive'].fillna(1.0).tolist()
_rcolors = ['seagreen' if v <= 1.0 else 'firebrick' for v in _ratios]

fig, axes = plt.subplots(1, 3, figsize=(22, 5))

axes[0].bar(_x - _width/2, _maes, _width, label='MAE', color='steelblue')
axes[0].bar(_x + _width/2, _mses, _width, label='MSE', color='darkorange')
axes[0].set_xticks(_x); axes[0].set_xticklabels(forecaster_names, rotation=15, ha='right')
axes[0].set_title(f'MAE & MSE — {ACTIVE_DGP}')
axes[0].set_ylabel('Error'); axes[0].legend()
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

axes[1].bar(_x, _ratios, color=_rcolors)
axes[1].axhline(1.0, color='black', linestyle='--', linewidth=1.2, label='Naive baseline')
axes[1].set_xticks(_x); axes[1].set_xticklabels(forecaster_names, rotation=15, ha='right')
axes[1].set_title('RMSE / Naive  —  green < 1 = beats Naive')
axes[1].set_ylabel('Ratio'); axes[1].legend()
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

results.plot_mse_over_time(rolling_window=ROLLING_WINDOW, ax=axes[2])
plt.tight_layout(); plt.show()

fig2, ax2 = plt.subplots(figsize=(14, 4))
results.plot_cumulative_error(ax=ax2)
plt.tight_layout(); plt.show()

fig3, ax3 = plt.subplots(figsize=(14, 4))
ax3.plot(origin_dates, actuals, label='Actual', color='black', linewidth=1.2)
for name in forecaster_names:
    preds = results.predictions[name].ravel()
    lw = 1.4 if name == 'TTM (granite)' else 0.8
    ls = '-'  if name == 'TTM (granite)' else '--'
    ax3.plot(origin_dates, preds, label=name, color=COLORS.get(name, 'steelblue'), linewidth=lw, linestyle=ls)
ax3.set_title(f'{ACTIVE_DGP} — {HORIZON}-step-ahead point forecasts')
ax3.legend(fontsize=8)
plt.tight_layout(); plt.show()